In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.chdir('/content/drive/MyDrive/Datascience')

In [ ]:
# !pip uninstall tensorflow -y

In [ ]:
!pip install tensorflow

In [ ]:
! pip install autokeras

In [ ]:
import os
import pandas as pd
import numpy as np
import cv2
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, confusion_matrix
import autokeras as ak

# Function to load and preprocess images
def load_and_preprocess_images(main_dir, csv_file, target_size=(128, 128)):
    # Load the CSV file
    data = pd.read_csv(csv_file)

    images = []
    labels = []

    for subdir in os.listdir(main_dir):
        sub_dir = os.path.join(main_dir, subdir)
        if os.path.isdir(sub_dir):
            for image_file in os.listdir(sub_dir):
                if image_file.endswith('.png'):  # Assuming images are in PNG format
                    image_path = os.path.join(sub_dir, image_file)
                    image = cv2.imread(image_path)
                    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert image to RGB
                    image = cv2.resize(image, target_size)  # Resize image
                    image = image.astype('float32') / 255.0  # Normalize pixel values

                    images.append(image)

                    # Get class label from CSV file based on image filename
                    image_code = os.path.splitext(image_file)[0]
                    label = data.loc[data['id_code'] == image_code, 'diagnosis'].values[0]
                    labels.append(label)

    images = np.array(images)
    labels = np.array(labels)

    return images, labels

# Directory containing subfolders with images
# main_dir = r'ret\dataset\gaussian_filtered_images'
main_dir = r'/content/drive/MyDrive/Datascience/gaussian_filtered_images/gaussian_filtered_images'
# CSV file containing image labels
# csv_file = r'ret\dataset\train.csv'
csv_file = r'/content/drive/MyDrive/Datascience/train.csv'

# Load and preprocess images with reduced size
images, labels = load_and_preprocess_images(main_dir, csv_file, target_size=(128, 128))

# Encode class labels if necessary
label_encoder = LabelEncoder()
labels_encoded = label_encoder.fit_transform(labels)

# Split the dataset into training and validation sets
train_images, val_images, train_labels, val_labels = train_test_split(images, labels_encoded, test_size=0.2, random_state=42)

# Initialize the ImageClassifier
clf = ak.ImageClassifier(overwrite=True, max_trials=1) # max_trials denotes the number of different models to try

# Fit the model
clf.fit(train_images, train_labels, epochs=20, validation_data=(val_images, val_labels))

# Evaluate the best model after search
loss, accuracy = clf.evaluate(val_images, val_labels)
print(f'Validation Loss: {loss:.4f}, Validation Accuracy: {accuracy:.4f}')

# Get the best performing model
model = clf.export_model()
model.summary()  # This will give you a summary of the best model architecture found by AutoKeras

# Define the model architecture
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(5, activation='softmax')  # Assuming 5 classes
])

# Compile the model
optimizer = Adam(learning_rate=0.0001)  # Lowering learning rate for better convergence
model.compile(optimizer=optimizer,
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Train the model
history = model.fit(train_images, train_labels, epochs=20, batch_size=64, validation_data=(val_images, val_labels))

# Evaluate the model on the validation set
val_loss, val_accuracy = model.evaluate(val_images, val_labels)
print(f'Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}')

# Make predictions on the validation set
val_predictions = model.predict(val_images)
val_predicted_labels = np.argmax(val_predictions, axis=1)

# Calculate additional evaluation metrics
print('Classification Report:')
print(classification_report(val_labels, val_predicted_labels))

print('Confusion Matrix:')
print(confusion_matrix(val_labels, val_predicted_labels))

# Save the model
#model.save('my_model12.h5')
